In [1]:
from coffea.nanoevents import NanoAODSchema
from coffea.dataset_tools import apply_to_fileset, max_chunks, max_files, preprocess

import dask

from analysis_tools.processors.processor_24_2_25 import MyProcessor
from dask.distributed import Client

/usr/local/lib/python3.12/site-packages/coffea/nanoevents/schemas/fcc.py:5: FutureWarning: In version 2025.1.0 (target date: 2024-12-31 11:59:59-06:00), this will be an error.
To raise these warnings as errors (and get stack traces to find out where they're called), run
    import warnings
    warnings.filterwarnings("error", module="coffea.*")
after the first `import coffea` or use `@pytest.mark.filterwarnings("error:::coffea.*")` in pytest.
Issue: coffea.nanoevents.methods.vector will be removed and replaced with scikit-hep vector. Nanoevents schemas internal to coffea will be migrated. Otherwise please consider using that package!.
  from coffea.nanoevents.methods import vector


In [2]:
import gzip
import json
import os

# Define the base directory where the preprocessed files are stored
base_dir = "/home/cms-jovyan/dwg_analysis/tools/preprocessing/preprocessed"
signal_sample = "2023_SlepSnu_MN1_270_100000_preprocessed_available.json.gz"
#signal_sample = "2023_ttbar_100000_preprocessed_available.json.gz"
background_sample = "2023_ttbar_100000_preprocessed_available.json.gz"
signal_file_path = os.path.join(base_dir, signal_sample)
background_file_path = os.path.join(base_dir, background_sample)
#print(preprocessed)

with gzip.open(signal_file_path, "rt") as file:
    signal_preprocessed_available = json.load(file)
with gzip.open(background_file_path, "rt") as file:
    background_preprocessed_available = json.load(file)

In [3]:
#client = Client("tls://localhost:8786")
#client

In [4]:
signal_test_preprocessed_files = max_files(signal_preprocessed_available, 5)
signal_test_preprocessed = max_chunks(signal_test_preprocessed_files, 1)

### SWITCH HERE ###

signal_reduced_computation = True

###################

# signal

if signal_reduced_computation:
    small_tg, small_rep = apply_to_fileset(
        data_manipulation=MyProcessor(),
        fileset=signal_test_preprocessed,
        schemaclass=NanoAODSchema,
        uproot_options={"allow_read_errors_with_report": (OSError, KeyError)},
    )
    signal_computed, rep = dask.compute(small_tg, small_rep)
    
else:
    full_tg, full_rep = apply_to_fileset(
        data_manipulation=MyProcessor(),
        fileset=signal_preprocessed_available,
        schemaclass=NanoAODSchema,
        uproot_options={"allow_read_errors_with_report": (OSError, KeyError)},
    )
    signal_computed, rep = dask.compute(full_tg, full_rep)


In [5]:
signal_computed

{'/SlepSnuCascade_MN1-270_MN2-280_MC1-275_TuneCP5_13p6TeV_madgraphMLM-pythia8/Run3Summer23BPixNanoAODv12-130X_mcRun3_2023_realistic_postBPix_v6-v3/NANOAODSIM': {'counts': {'total_entries': 75578,
   'total_entries_oldmethod': 149216,
   'total_ele': 111766,
   'test_1': <Array [1, 0, 2, 1, 1, 1, 3, 2, ..., 1, 1, 1, 0, 0, 1, 2] type='149216 * int64'>,
   'test_2': 75578},
  'pt_binned': {},
  'calculations': {},
  'plots': {},
  'tests': {}}}

In [6]:
sig_results = signal_computed['/SlepSnuCascade_MN1-270_MN2-280_MC1-275_TuneCP5_13p6TeV_madgraphMLM-pythia8/Run3Summer23BPixNanoAODv12-130X_mcRun3_2023_realistic_postBPix_v6-v3/NANOAODSIM']
#sig_results = signal_computed['/TTto2L2Nu_TuneCP5_13p6TeV_powheg-pythia8/Run3Summer23NanoAODv12-130X_mcRun3_2023_realistic_v14-v2/NANOAODSIM']

Shooting for ~ 459524 events for ttbar

In [8]:
sig_results

{'counts': {'total_entries': 75578,
  'total_entries_oldmethod': 149216,
  'total_ele': 111766,
  'test_1': <Array [1, 0, 2, 1, 1, 1, 3, 2, ..., 1, 1, 1, 0, 0, 1, 2] type='149216 * int64'>,
  'test_2': 75578},
 'pt_binned': {},
 'calculations': {},
 'plots': {},
 'tests': {}}

75578